# ClimaTwin India — Colab GPU Training

**AI-Powered Digital Twin of India's Climate** (ISRO PoC). This notebook runs the
heavy training on a free Colab **GPU** (your MacBook Air is fine for the demo, not
for full training). It will:

1. Build the data cube from **real IMD national data** (+ INSAT LST channel).
2. Train the **ConvLSTM** forecaster (full 120-epoch run, CUDA auto-detected).
3. Train the **SR-CNN** rainfall downscaler.
4. **Validate** against the persistence + climatology baselines.
5. Package the trained artifacts so you download them back into the local repo.

> **FIRST:** `Runtime` → `Change runtime type` → **Hardware accelerator: GPU** → Save.

Everything here uses Colab's **own** Python (`!python` / `!pip`) — there is no `.venv`
on Colab. (`.venv` is only for your local MacBook.)

## 1. Check the GPU
Confirm a GPU is attached and visible to PyTorch.

In [ ]:
!nvidia-smi

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Get the code
Two options — **(b) upload bundle is the default**.

**(a) Clone from GitHub** (only if you've pushed the repo). Edit the URL, then run:
```python
# !git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git climatwin
# %cd climatwin
```

**(b) Upload bundle (default).** On your Mac first run `bash scripts/make_colab_bundle.sh`
to produce `climatwin_bundle.zip`, then run the cell below and pick that file.

In [ ]:
# Option (a) — clone from GitHub (uncomment + set your repo URL):
# !git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git climatwin
# %cd climatwin

In [ ]:
# Option (b) DEFAULT — upload climatwin_bundle.zip from your Mac:
from google.colab import files
uploaded = files.upload()   # choose climatwin_bundle.zip
!unzip -q -o climatwin_bundle.zip -d climatwin
%cd climatwin
!ls

## 3. Install the extra deps
Colab already ships torch, tensorflow, numpy, pandas, scipy, scikit-learn and
matplotlib — we do **not** reinstall those. Only the geo/climate stack is missing.

In [ ]:
!pip install -q imdlib xarray netCDF4 cftime rioxarray rasterio remotezip

## 4. Build the data cube (real IMD + INSAT LST)
Real IMD gridded rainfall/temperature downloads work on Colab. `--with-lst` fuses
the INSAT LST channel. Writes `data/twin_cube.nc` and `data/norm_stats.json`
(normalization computed on **train years only** — no leakage).

In [ ]:
import os
# If a prebuilt cube was shipped in the bundle (e.g. one built locally WITH real
# INSAT LST via `make data-lst`), train on it directly. Otherwise build from IMD here.
if os.path.exists('data/twin_cube.nc'):
    print('Using prebuilt cube data/twin_cube.nc (shipped in bundle) — skipping build.')
    import xarray as xr
    d = xr.open_dataset('data/twin_cube.nc')
    print('  data_source:', d.attrs.get('data_source'), '| lst_source:', d.attrs.get('lst_source'),
          '| vars:', list(d.data_vars))
else:
    !python -m data.build_cube --source imd --with-lst

## 5. Train the ConvLSTM (full GPU run)
120 epochs on the GPU. Rainfall is trained in log1p space with a wet-cell-weighted
loss. Saves `models/checkpoints/convlstm.pt` + `data/norm_stats.json` use.

In [ ]:
!python -m models.train --epochs 120

## 6. Train the SR-CNN downscaler
Super-resolution CNN for rainfall (coarse → 0.25°). 200 epochs.

> Note: requires `models/downscale.py` (P1 upgrade). If it isn't in your bundle yet,
> skip this cell — the ConvLSTM above is the P0 core.

In [ ]:
!python -m models.downscale --var rainfall --epochs 200

## 6b. Train the diffusion downscaler (CorrDiff-style · INDmet 0.05°)
A **residual diffusion** model (NVIDIA Earth-2 CorrDiff-style) that super-resolves rainfall 
0.25° → 0.05° using real **INDmet** 5 km truth, and samples an **ensemble** of plausible sharp 
fields. Scored on **spatial/spectral skill** (power-spectra, FSS, CRPS) — where generative 
downscaling wins and a deterministic SR-CNN can't (the RMSE “double-penalty”).

> Heavy: ~120 epochs on the GPU (a few minutes on a T4 for the 40×60 region). The 0.05° cube 
ships in the `--with-data` bundle; otherwise the cell downloads it from Zenodo first.

In [ ]:
# CorrDiff-style residual diffusion downscaler on the real INDmet 0.05° cube.
import os
if not os.path.exists('data/indmet_cube_005.nc'):
    print('INDmet 0.05° cube not in bundle — downloading + building (~6 GB rainfall from Zenodo)…')
    !python -m data.ingest_indmet --vars rainfall --years 2000 2023
!python -m models.diffusion_downscale --epochs 120

## 7. Validate against the baselines
RMSE/MAE/corr + POD/FAR/CSI on the **temporal test split**, ConvLSTM vs persistence
vs climatology. Writes `models/validation_metrics.json`.

In [ ]:
!python -m models.validate

In [ ]:
import json
with open('models/validation_metrics.json') as f:
    m = json.load(f)
print('summary_rmse (by horizon):')
print(json.dumps(m.get('summary_rmse', {}), indent=2))

## 8. Package + download the trained artifacts
Bundle the cube, norm stats, checkpoints and metrics so you can drop them into your
local repo and serve the demo offline.

In [ ]:
import os
!rm -f climatwin_trained.zip
paths = [
    'data/twin_cube.nc',
    'data/norm_stats.json',
    'models/validation_metrics.json',
]
paths = [p for p in paths if os.path.exists(p)]
# add all checkpoints
ckpt = 'models/checkpoints'
if os.path.isdir(ckpt):
    paths += [os.path.join(ckpt, f) for f in os.listdir(ckpt) if not f.startswith('.')]
print('zipping:', paths)
import subprocess
subprocess.run(['zip', '-r', '-q', 'climatwin_trained.zip', *paths], check=True)
!ls -lh climatwin_trained.zip

In [ ]:
from google.colab import files
files.download('climatwin_trained.zip')

## Done — wire it back into the local repo
On your Mac, from the repo root:
```bash
unzip -o ~/Downloads/climatwin_trained.zip -d .
# this restores data/twin_cube.nc, data/norm_stats.json,
# models/checkpoints/*, models/validation_metrics.json
make serve   # FastAPI on http://127.0.0.1:8000  (fully offline, from cache)
```
The demo now runs offline against the GPU-trained checkpoint. 🌦️